### Overview
* The objective of this notebook is to fine-tune a **pre-trained Japanese BERT model (`cl-tohoku/bert-base-japanese-v3`)** on Japanese review texts (`review_clean_p.csv`, with `text` column and `label` column N/P) to build and evaluate a binary sentiment classification model that classifies reviews as Negative (0) or Positive (1).

### Prerequisites  
* The CSV file is encoded in **UTF-8**, and the main columns are `text` (body text) and `label` (N=Negative, P=Positive).  
* Any extra index columns (e.g., `Unnamed: 0`) will be automatically removed.

### Preparation
* To perform processing using a **GPU**, please set the hardware accelerator to **GPU** by navigating to "Runtime" -> "Change runtime type" from the top menu bar in Google Colab.

##### 1. Environment Setup
* Install Transformers, Datasets, scikit-learn, and Matplotlib.  
* Use the pre-trained Japanese BERT model (**`cl-tohoku/bert-base-japanese-v3`**).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/test/  # Modify the path or directory to match your environment

Mounted at /content/drive
/content/drive/MyDrive/Deep learning/sc_nlp


In [ ]:

%%capture
# If running on Colab, uncomment the line below to upgrade pip
# !pip install -U pip

!pip install -U transformers datasets accelerate scikit-learn matplotlib pandas sentencepiece


##### 2. Imports & Setting Random Seeds

In [ ]:

import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score, precision_recall_fscore_support)

from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

# Seed value, initial random state
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))


PyTorch version: 2.11.0+cu130
CUDA available: True
GPU name: Tesla T4


##### 3. Data Loading (UTF-8)
* Load `review_clean_p.csv` using UTF-8 encoding and use `text` and `label`. Convert `label` from 'N' -> 0 (Negative) and 'P' -> 1 (Positive).  

In [ ]:

# Change path if necessary
CSV_PATH = '/content/drive/MyDrive/test/review_clean_p.csv'

# Load in UTF-8 (fallback to UTF-8-SIG if decoding fails)
try:
    df = pd.read_csv(CSV_PATH, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')

# Drop columns that clearly look like index columns
drop_cols = [c for c in df.columns if c.lower().startswith('unnamed')]
if drop_cols:
    df = df.drop(columns=drop_cols)

# Basic checks
assert 'text' in df.columns, f"'text' column not found. Found: {df.columns.tolist()}"
assert 'label' in df.columns, f"'label' column not found. Found: {df.columns.tolist()}"

# Convert labels to integers: N=0, P=1
label_map = {'N': 0, 'P': 1, 0: 0, 1: 1}
df['label'] = df['label'].map(label_map)

# If NaNs remain in label column, print warning / inspect unique values
if df['label'].isna().any():
    print('Warning: Unknown labels found. Showing unique values:')
    print(df['label'].unique())
    # Add custom mapping here if necessary

# Drop rows with missing values (NA)
before = len(df)
df = df.dropna(subset=['text', 'label'])
after = len(df)
print(f'Read {before} rows, using {after} after dropping NA.')

# Display a small preview
df.head(3)


Read 1000 rows, using 1000 after dropping NA.


##### Dataset
* We use a sentiment analysis dataset consisting of labels (replacing positive/negative categories with labels 0 and 1) and sentences as shown above.

##### 4. Train/Test Data Split
* Use stratified split to maintain class label proportions (test_size=0.2).  

In [ ]:

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df['label']
)
print(train_df.shape, test_df.shape)
train_df['label'].value_counts(normalize=True), test_df['label'].value_counts(normalize=True)


(800, 2) (200, 2)


##### 5. Tokenization & Dataset Preparation
* Model: `cl-tohoku/bert-base-japanese-v3` (Japanese BERT).  
* Preprocessing: Automate truncation and padding.

In [ ]:

!pip install fugashi ipadic unidic-lite

MODEL_ID = 'cl-tohoku/bert-base-japanese-v3'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Convert to HF Datasets format
ds_train = Dataset.from_pandas(train_df[['text','label']])
ds_test  = Dataset.from_pandas(test_df[['text','label']])

max_length = 256

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding=False, max_length=max_length)

ds_train = ds_train.map(tokenize, batched=True, remove_columns=['text'])
ds_test  = ds_test.map(tokenize, batched=True, remove_columns=['text'])

collator = DataCollatorWithPadding(tokenizer=tokenizer)


##### 6. Training / Fine-tuning
* Train for 2-3 epochs. Use a small batch size to reduce training time and set the learning rate around 2e-5.  

In [ ]:

num_labels = 2
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=num_labels)

args = TrainingArguments(
    output_dir='outputs',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=50,
    load_best_model_at_end=False,
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    data_collator=collator,
)

trainer.train()


##### 7. Evaluation Metrics
* Display Accuracy, Precision, Recall, and F1-score.  

In [ ]:

# Get predictions
pred = trainer.predict(ds_test)
logits = pred.predictions
y_prob = torch.softmax(torch.tensor(logits), dim=-1).numpy()
y_pred = y_prob.argmax(axis=1)
y_true = np.array(test_df['label'].tolist())

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')

print('Accuracy :', round(acc, 4))
print('Precision:', round(prec, 4))
print('Recall   :', round(rec, 4))
print('F1-score :', round(f1, 4))

print('Classification report:')
print(classification_report(y_true, y_pred, target_names=['Negative (0)','Positive (1)']))


##### 8. Normalized Confusion Matrix
* Display the matrix normalized along rows (true labels) to observe accuracy per class.  

In [ ]:

cm = confusion_matrix(y_true, y_pred, normalize='true')
fig, ax = plt.subplots(figsize=(5, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative','Positive'])
disp.plot(cmap='Blues', ax=ax, values_format='.2f', colorbar=False)
ax.set_title('Normalized Confusion Matrix (row-normalized)')
plt.tight_layout()
plt.show()


##### 9. Inference Demo
* Perform sentiment prediction on arbitrary sample sentences.  

In [ ]:

examples = [
    '最高でした。もう一度見たい！',
    '残念。期待外れでつまらなかった。',
]

enc = tokenizer(examples, return_tensors='pt', truncation=True, padding=True, max_length=256)
model.eval()
with torch.no_grad():
    out = model(**{k: v.to(model.device) for k, v in enc.items()})
probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
labels = probs.argmax(axis=1)

for sent, p, lab in zip(examples, probs, labels):
    print(f'{sent} -> pred={lab} (0=Negative, 1=Positive), P(neg,pos)={p.round(3)}')
